In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
CATALOG = "clinical_trial_supply_chain"

BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

SOURCE_TABLE = f"{BRONZE}.bronze_site_master"
TARGET_TABLE = f"{SILVER}.dim_site_scd2"

In [0]:
bronze_df = (
    spark.table(SOURCE_TABLE)
)

latest_ts = (
    bronze_df
    .groupBy("site_id")
    .agg(
        F.max("last_updated_timestamp")
        .alias("latest_timestamp")
    )
)

site_snapshot_df = (
    bronze_df.alias("b")
    .join(
        latest_ts.alias("l"),
        (
            (F.col("b.site_id") == F.col("l.site_id"))
            &
            (
                F.col("b.last_updated_timestamp")
                ==
                F.col("l.latest_timestamp")
            )
        ),
        "inner"
    )
    .select(
        "b.site_id",
        "b.site_name",
        "b.study_id",
        "b.country_code",
        "b.region",
        "b.site_status",
        "b.site_manager"
    )
)

In [0]:
current_df = (
    spark.table(TARGET_TABLE)
    .filter(F.col("is_current") == True)
)

In [0]:
changes_df = (
    site_snapshot_df.alias("s")
    .join(
        current_df.alias("t"),
        "site_id",
        "inner"
    )
    .filter(
        (F.col("s.site_status") != F.col("t.site_status"))
        |
        (F.col("s.site_manager") != F.col("t.site_manager"))
    )
    .select("s.*")
)

In [0]:
target = DeltaTable.forName(
    spark,
    TARGET_TABLE
)

(
    target.alias("t")
    .merge(
        changes_df.alias("s"),
        """
        t.site_id = s.site_id
        AND t.is_current = true
        """
    )
    .whenMatchedUpdate(
        set={
            "effective_end_date": "current_timestamp()",
            "is_current": "false"
        }
    )
    .execute()
)

In [0]:
new_sites_df = (
    site_snapshot_df.alias("s")
    .join(
        current_df.alias("t"),
        "site_id",
        "left_anti"
    )
)

insert_df = (
    new_sites_df
    .unionByName(changes_df)
)

In [0]:
insert_df = (
    insert_df
    .withColumn(
        "effective_start_date",
        F.current_timestamp()
    )
    .withColumn(
        "effective_end_date",
        F.lit("9999-12-31 00:00:00")
        .cast("timestamp")
    )
    .withColumn(
        "is_current",
        F.lit(True)
    )
)

(
    insert_df.write
    .mode("append")
    .saveAsTable(TARGET_TABLE)
)

In [0]:
display(
    spark.sql(
        f"""
        SELECT *
        FROM {TARGET_TABLE}
        WHERE is_current = true
        ORDER BY site_id
        """
    )
)